# 03 — Parametric benefit atlas

`studies/atlas.py` sweeps the design space and caches every sweep as Parquet
under `data/` (committed, so this notebook runs from a fresh clone without
recomputing). To regenerate from scratch (~2 min):

```bash
uv run python studies/atlas.py
```

Cells below reproduce the headline maps; each grid cell carries a `valid`
flag from the boundary-layer solver so separated corners are shown greyed,
never silently interpolated.


In [ ]:
# resolve the repo root so the notebook runs from notebooks/ or the repo root
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "studies"))
import plotstyle

plotstyle.apply()
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA = ROOT / "data"
atlas = pd.read_parquet(DATA / "atlas_fphi_fpr.parquet")
print(f"{len(atlas)} grid cells, {int((~atlas.valid).sum())} flagged invalid")
atlas.head()


In [ ]:
# Headline heatmap: subsystem PSC over (f_phi, FPR)
piv = atlas.pivot(index="fpr", columns="f_phi", values="psc")
mask = atlas.pivot(index="fpr", columns="f_phi", values="valid")
z = np.where(mask.values, piv.values * 100, np.nan)

fig, ax = plt.subplots(figsize=(6, 3.5))
pc = ax.pcolormesh(piv.columns.values, piv.index.values, z,
                   cmap="viridis", shading="auto")
fig.colorbar(pc, label="subsystem PSC [%]")
ax.set_xlabel(r"$f_\Phi$"); ax.set_ylabel("FPR")
ax.set_title("paper Fig. 4 (grey/blank = separated, flagged invalid)");


In [ ]:
# Diminishing returns: normalized saving saturates with ingestion fraction
sat = pd.read_parquet(DATA / "atlas_saturation.parquet")
fig, ax = plt.subplots(figsize=(6, 3))
for fpr, g in sat.groupby("fpr"):
    ax.plot(g.f_phi, g.saving_norm, label=f"FPR = {fpr:.2f}")
ax.set_xlabel(r"$f_\Phi$"); ax.set_ylabel("saving / max saving")
ax.legend(); ax.set_title("saturation of the BLI benefit (paper Fig. 5)");


In [ ]:
# Robustness of the benefit across flight condition and geometry
mach = pd.read_parquet(DATA / "atlas_mach_alt.parquet")
fin = pd.read_parquet(DATA / "atlas_fineness.parquet")

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for alt, g in mach.groupby("altitude"):
    axs[0].plot(g.mach, g.psc * 100, marker="o", ms=3,
                label=f"{alt/1e3:.1f} km")
axs[0].set_xlabel("Mach"); axs[0].set_ylabel("subsystem PSC [%]")
axs[0].legend(fontsize=8, title="altitude")
axs[1].plot(fin.fineness, fin.psc * 100, marker="s", ms=3, color="C1")
axs[1].set_xlabel("fuselage fineness L/D"); axs[1].set_ylabel("subsystem PSC [%]")
fig.suptitle("PSC vs flight condition and fuselage slenderness", y=1.02);


`atlas_design_curves.parquet` (thrust-share sweeps at fixed net force) feeds
the right panel of paper Fig. 5; see `studies/figures.py::fig5_saturation`.
